In [1]:
import pandas as pd
import numpy as np
import pm4py

In [2]:
def returnPrototypeSeq(dirProt,log):
    protInfo=pd.read_csv(dirProt)
    case_id_prot=protInfo["case:concept:name"][0]
    activity_sequence=log[log["case:concept:name"]==case_id_prot]["concept:name"].values
    return activity_sequence

In [3]:
sepsis_log=pm4py.read_xes("./data/sepsis/sepsis.xes")

c:\Users\ccagu\anaconda3\envs\naturalexamples\lib\site-packages\pm4py\util\dt_parsing\parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
c:\Users\ccagu\anaconda3\envs\naturalexamples\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 1050/1050 [00:01<00:00, 903.63it/s]


In [4]:
sirs_false_sequence=returnPrototypeSeq(dirProt="./results/Ours/sepsis/05-18-2026_17-03-53/best_predicted_instance_classSIRS-False.csv",
                                       log=sepsis_log)

In [5]:
sirs_true_sequence=returnPrototypeSeq(dirProt="./results/Ours/sepsis/05-18-2026_17-03-53/best_predicted_instance_classSIRS-True.csv",
                                       log=sepsis_log)

In [8]:
bc_sirs_false_true=pd.read_csv("./results/Ours/sepsis/02-25-2026_12-28-00/boundaryCases_ClassesSIRS-False-SIRS-True.csv")

In [24]:
boundary_case_sirs_false=sepsis_log[sepsis_log["case:concept:name"]==bc_sirs_false_true["case:concept:name"][0]]["concept:name"].values

In [25]:
boundary_case_sirs_true=sepsis_log[sepsis_log["case:concept:name"]==bc_sirs_false_true["case:concept:name"][0]]["concept:name"].values

In [30]:
boundary_case_sirs_false

array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
       'Leucocytes', 'Admission NC', 'Leucocytes', 'CRP', 'Leucocytes',
       'Leucocytes', 'CRP', 'CRP', 'Leucocytes', 'Release A', 'CRP',
       'Leucocytes'], dtype=object)

In [31]:
def getSIRSPrototypesDir(directory,log, regular_prot_sirs_true, regular_prot_sirs_false, regular_bc_false, regular_bc_true):
    sirs_false_sequence_reduced=returnPrototypeSeq(dirProt="./results/Ours/sepsis/"+directory+"/best_predicted_instance_classSIRS-False.csv",
                                                   log=log)
    sirs_true_sequence_reduced=returnPrototypeSeq(dirProt="./results/Ours/sepsis/"+directory+"/best_predicted_instance_classSIRS-True.csv",
                                                  log=log)
    
    bc_sirs_false_true_ids=pd.read_csv("./results/Ours/sepsis/"+directory+"/boundaryCases_ClassesSIRS-False-SIRS-True.csv")
    boundary_case_sirs_false_seq=log[log["case:concept:name"]==bc_sirs_false_true_ids["case:concept:name"][0]]["concept:name"].values
    boundary_case_sirs_true_seq=log[log["case:concept:name"]==bc_sirs_false_true_ids["case:concept:name"][1]]["concept:name"].values


    are_prots_sirs_false_equal=np.array_equal(regular_prot_sirs_false, sirs_false_sequence_reduced)
    are_prots_sirs_true_equal=np.array_equal(regular_prot_sirs_true, sirs_true_sequence_reduced)
    are_bc_sirs_false_equal=np.array_equal(regular_bc_false, boundary_case_sirs_false_seq)
    are_bc_sirs_true_equal=np.array_equal(regular_bc_true, boundary_case_sirs_true_seq)

    results={}
    results["SIRS-false"]=are_prots_sirs_false_equal
    results["SIRS-True"]=are_prots_sirs_true_equal
    results["bc-SIRS-false"]=are_bc_sirs_false_equal
    results["bc-SIRS-true"]=are_bc_sirs_true_equal

    results["SIRS-false_reduced-prot"]=sirs_false_sequence_reduced 
    results["SIRS-true_reduced-prot"]=sirs_true_sequence_reduced
    results["SIRS-true-bc-seq"]=boundary_case_sirs_true_seq
    results["SIRS-false-bc-seq"]=boundary_case_sirs_false_seq

    return results



In [33]:
getSIRSPrototypesDir(directory="declare_rules-XGB-limited_tr_data0.2505-22-2026_15-47-45",
                     log=sepsis_log,
                     regular_prot_sirs_true=sirs_true_sequence, 
                     regular_prot_sirs_false=sirs_false_sequence,
                     regular_bc_false=boundary_case_sirs_false,
                     regular_bc_true=boundary_case_sirs_true)

{'SIRS-false': False,
 'SIRS-True': False,
 'bc-SIRS-false': False,
 'bc-SIRS-true': False,
 'SIRS-false_reduced-prot': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
        'Admission NC', 'Admission NC', 'Release A', 'Return ER'],
       dtype=object),
 'SIRS-true_reduced-prot': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
        'Leucocytes', 'IV Antibiotics'], dtype=object),
 'SIRS-true-bc-seq': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'Leucocytes',
        'CRP', 'LacticAcid'], dtype=object),
 'SIRS-false-bc-seq': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
        'LacticAcid', 'Leucocytes'], dtype=object)}

In [35]:
getSIRSPrototypesDir(directory="declare_rules-XGB-limited_tr_data0.505-22-2026_15-48-27",
                     log=sepsis_log,
                     regular_prot_sirs_true=sirs_true_sequence, 
                     regular_prot_sirs_false=sirs_false_sequence,                     regular_bc_false=boundary_case_sirs_false,
                     regular_bc_true=boundary_case_sirs_true)

{'SIRS-false': True,
 'SIRS-True': False,
 'bc-SIRS-false': False,
 'bc-SIRS-true': False,
 'SIRS-false_reduced-prot': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage'], dtype=object),
 'SIRS-true_reduced-prot': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
        'Leucocytes', 'IV Antibiotics'], dtype=object),
 'SIRS-true-bc-seq': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'Leucocytes',
        'CRP', 'LacticAcid'], dtype=object),
 'SIRS-false-bc-seq': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
        'LacticAcid', 'Leucocytes'], dtype=object)}

In [36]:
getSIRSPrototypesDir(directory="declare_rules-XGB-limited_tr_data0.7505-22-2026_15-48-58",
                     log=sepsis_log,
                     regular_prot_sirs_true=sirs_true_sequence, 
                     regular_prot_sirs_false=sirs_false_sequence,                     regular_bc_false=boundary_case_sirs_false,
                     regular_bc_true=boundary_case_sirs_true)

{'SIRS-false': True,
 'SIRS-True': False,
 'bc-SIRS-false': False,
 'bc-SIRS-true': False,
 'SIRS-false_reduced-prot': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage'], dtype=object),
 'SIRS-true_reduced-prot': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'Leucocytes',
        'LacticAcid', 'CRP', 'IV Liquid', 'IV Antibiotics'], dtype=object),
 'SIRS-true-bc-seq': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'Leucocytes',
        'CRP', 'LacticAcid'], dtype=object),
 'SIRS-false-bc-seq': array(['ER Registration', 'ER Triage', 'ER Sepsis Triage', 'CRP',
        'LacticAcid', 'Leucocytes'], dtype=object)}

In [37]:
#########################################################################################################################################################################################################################################################

In [38]:
rtfm_log=pm4py.read_xes("./Data/road_traffic/Road_Traffic_Fine_Management_Process.xes")

parsing log, completed traces :: 100%|██████████| 150370/150370 [00:59<00:00, 2520.56it/s]


In [ ]:
prot_collected=returnPrototypeSeq(dirProt="./results/Ours/rtfm/02-25-2026_15-59-27/best_predicted_instance_classcollected.csv",log=rtfm_log)

In [ ]:
prot_dismissed=returnPrototypeSeq(dirProt="./results/Ours/rtfm/02-25-2026_15-59-27/best_predicted_instance_classdismissed.csv",log=rtfm_log)

In [ ]:
prot_fully_paid=returnPrototypeSeq(dirProt="./results/Ours/rtfm/02-25-2026_15-59-27/best_predicted_instance_classfully_paid.csv",log=rtfm_log)

In [42]:
prot_unresolved=returnPrototypeSeq(dirProt="./results/Ours/rtfm/02-25-2026_15-59-27/best_predicted_instance_classunresolved.csv",log=rtfm_log)

In [43]:
bc_dismissed_unresolved=pd.read_csv("./results/Ours/rtfm/02-25-2026_15-59-27/boundaryCases_Classesdismissed-unresolved.csv")

In [48]:
bc_du_dismissed=rtfm_log[rtfm_log["case:concept:name"]==bc_dismissed_unresolved["case:concept:name"][0]]["concept:name"].values

In [49]:
bc_du_unresolved=rtfm_log[rtfm_log["case:concept:name"]==bc_dismissed_unresolved["case:concept:name"][1]]["concept:name"].values

In [44]:
bc_fullypaid_unresolved=pd.read_csv("./results/Ours/rtfm/02-25-2026_15-59-27/boundaryCases_Classesfully_paid-unresolved.csv")

In [50]:
bc_fpu_fully_paid=rtfm_log[rtfm_log["case:concept:name"]==bc_fullypaid_unresolved["case:concept:name"][0]]["concept:name"].values

In [51]:
bc_fpu_unresolved=rtfm_log[rtfm_log["case:concept:name"]==bc_fullypaid_unresolved["case:concept:name"][1]]["concept:name"].values

In [67]:
def getRTFM_infoDir(directory,log, regular_prot_collected, regular_prot_dismissed, regular_prot_fully_paid, regular_prot_unresolved, regular_bc_dissunr_dismissed_case, regular_bc_dissunr_unresolved_case, regular_bc_fpu_fully_paid, regular_bc_fpu_unresolved):
    collected_sequence_reduced=returnPrototypeSeq(dirProt="./results/Ours/rtfm/"+directory+"/best_predicted_instance_classcollected.csv",
                                                  log=log)
    
    dismissed_sequence_reduced=returnPrototypeSeq(dirProt="./results/Ours/rtfm/"+directory+"/best_predicted_instance_classdismissed.csv",
                                                  log=log)
    
    fully_paid_sequence_reduced=returnPrototypeSeq(dirProt="./results/Ours/rtfm/"+directory+"/best_predicted_instance_classfully_paid.csv",
                                                   log=log)
    
    unresolved_sequence_reduced=returnPrototypeSeq(dirProt="./results/Ours/rtfm/"+directory+"/best_predicted_instance_classunresolved.csv",
                                                   log=log)
    
    are_prots_unresolved_equal=np.array_equal(regular_prot_unresolved,unresolved_sequence_reduced)
    are_prots_fully_paid_equal=np.array_equal(regular_prot_fully_paid,fully_paid_sequence_reduced)
    are_prots_dimissed_equal=np.array_equal(regular_prot_dismissed,dismissed_sequence_reduced)
    are_prots_collected_equal=np.array_equal(regular_prot_collected,collected_sequence_reduced)

    bc_dismissed_unresolved_reduced=pd.read_csv("./results/Ours/rtfm/"+directory+"/boundaryCases_Classesdismissed-unresolved.csv")
    bc_du_dismissed_reduced=log[log["case:concept:name"]==bc_dismissed_unresolved_reduced["case:concept:name"][0]]["concept:name"].values
    bc_du_unresolved_reduced=log[log["case:concept:name"]==bc_dismissed_unresolved_reduced["case:concept:name"][1]]["concept:name"].values
    boundary_sequences_dismissedunresolved={"dismissed":bc_du_dismissed_reduced, "unresolved":bc_du_unresolved_reduced}

    are_bc_dissunr_dismissed_equal=np.array_equal(regular_bc_dissunr_dismissed_case,bc_du_dismissed_reduced)
    are_bc_dissunr_unresolved_equal=np.array_equal(regular_bc_dissunr_unresolved_case,bc_du_unresolved_reduced)

    bc_fully_paid_unresolved_reduced=pd.read_csv("./results/Ours/rtfm/"+directory+"/boundaryCases_Classesfully_paid-unresolved.csv")
    bc_fullypaidunresolv_fullypaid_reduced=log[log["case:concept:name"]==bc_fully_paid_unresolved_reduced["case:concept:name"][0]]["concept:name"].values
    bc_fullypaidunresolv_unresolved_reduced=log[log["case:concept:name"]==bc_fully_paid_unresolved_reduced["case:concept:name"][1]]["concept:name"].values
    boundary_sequences_fullypaidunresolved={"fully_paid":bc_fullypaidunresolv_fullypaid_reduced, "unresolved":bc_fullypaidunresolv_unresolved_reduced}

    are_bc_fullypaidunresolved_fullypaid_equal=np.array_equal(bc_fullypaidunresolv_fullypaid_reduced,regular_bc_fpu_fully_paid)
    are_bc_fullypaidunresolved_unresolv_equal=np.array_equal(bc_fullypaidunresolv_unresolved_reduced,regular_bc_fpu_unresolved)



    results={}
    results["are_prots_unresolved_equal?"]=are_prots_unresolved_equal
    results["are_prots_fully_paid_equal?"]=are_prots_fully_paid_equal
    results["are_prots_dismissed_equal?"]=are_prots_dimissed_equal
    results["are_prots_collected_equal?"]=are_prots_collected_equal
    results["are_dismissed_cases_of_bc_dissunr_equal?"]=are_bc_dissunr_dismissed_equal
    results["are_unresolved_cases_of_bc_dissunr_equal?"]=are_bc_dissunr_unresolved_equal
    results["are_fully_paid_cases_of_bc_fullypaidunresolv_equal?"]=are_bc_fullypaidunresolved_fullypaid_equal
    results["are_unresolved_cases_of_bc_fullypaidunresolv_equal?"]=are_bc_fullypaidunresolved_unresolv_equal

    return results

In [68]:
getRTFM_infoDir(directory="declare_rules-XGB-limited_tr_data0.505-22-2026_15-55-18",
                log=rtfm_log,
                regular_prot_dismissed=prot_dismissed,
                regular_prot_collected=prot_collected,
                regular_prot_fully_paid=prot_fully_paid,
                regular_prot_unresolved=prot_unresolved,
                regular_bc_dissunr_dismissed_case=bc_du_dismissed,
                regular_bc_dissunr_unresolved_case=bc_du_unresolved,
                regular_bc_fpu_fully_paid=bc_fpu_fully_paid,
                regular_bc_fpu_unresolved=bc_fpu_unresolved)

{'are_prots_unresolved_equal?': True,
 'are_prots_fully_paid_equal?': True,
 'are_prots_dismissed_equal?': False,
 'are_prots_collected_equal?': True,
 'are_dismissed_cases_of_bc_dissunr_equal?': True,
 'are_unresolved_cases_of_bc_dissunr_equal?': False,
 'are_fully_paid_cases_of_bc_fullypaidunresolv_equal?': False,
 'are_unresolved_cases_of_bc_fullypaidunresolv_equal?': False}

In [69]:
getRTFM_infoDir(directory="declare_rules-XGB-limited_tr_data0.2505-22-2026_16-06-58",
                log=rtfm_log,
                regular_prot_dismissed=prot_dismissed,
                regular_prot_collected=prot_collected,
                regular_prot_fully_paid=prot_fully_paid,
                regular_prot_unresolved=prot_unresolved,
                regular_bc_dissunr_dismissed_case=bc_du_dismissed,
                regular_bc_dissunr_unresolved_case=bc_du_unresolved,
                regular_bc_fpu_fully_paid=bc_fpu_fully_paid,
                regular_bc_fpu_unresolved=bc_fpu_unresolved)

{'are_prots_unresolved_equal?': True,
 'are_prots_fully_paid_equal?': True,
 'are_prots_dismissed_equal?': False,
 'are_prots_collected_equal?': True,
 'are_dismissed_cases_of_bc_dissunr_equal?': True,
 'are_unresolved_cases_of_bc_dissunr_equal?': True,
 'are_fully_paid_cases_of_bc_fullypaidunresolv_equal?': False,
 'are_unresolved_cases_of_bc_fullypaidunresolv_equal?': False}

In [70]:
getRTFM_infoDir(directory="declare_rules-XGB-limited_tr_data0.7505-22-2026_16-03-53",
                log=rtfm_log,
                regular_prot_dismissed=prot_dismissed,
                regular_prot_collected=prot_collected,
                regular_prot_fully_paid=prot_fully_paid,
                regular_prot_unresolved=prot_unresolved,
                regular_bc_dissunr_dismissed_case=bc_du_dismissed,
                regular_bc_dissunr_unresolved_case=bc_du_unresolved,
                regular_bc_fpu_fully_paid=bc_fpu_fully_paid,
                regular_bc_fpu_unresolved=bc_fpu_unresolved)

{'are_prots_unresolved_equal?': True,
 'are_prots_fully_paid_equal?': True,
 'are_prots_dismissed_equal?': False,
 'are_prots_collected_equal?': True,
 'are_dismissed_cases_of_bc_dissunr_equal?': False,
 'are_unresolved_cases_of_bc_dissunr_equal?': True,
 'are_fully_paid_cases_of_bc_fullypaidunresolv_equal?': False,
 'are_unresolved_cases_of_bc_fullypaidunresolv_equal?': False}